# Germline Whole Exome Sequencing (WES): Variant Annotation and Filtering

**Author:** Eman Koosehlar

**Notebook:** 2 of the series  

**Version:** 1.5

**Last Updated:** July 2026

**License:** MIT License


---

### Runtime Requirements

This notebook is designed to run on the default Google Colab runtime.

**Recommended runtime**
- Runtime type: `CPU`

No GPU or TPU is required.

---

# Learning Journey

This notebook is the second part of the **Germline Whole Exome Sequencing (WES) Analysis Series**.

In  [**Notebook 1: Germline WES Variant Calling**](https://github.com/namia47/WES_Analysis_Tutorial/notebooks/01_Germline_WES_Variant_Calling.ipynb), we generated a high-confidence germline VCF through quality control, read alignment, variant calling, and hard filtering. In this notebook, we take the next step by transforming those variants into biologically meaningful information through **variant annotation** and **initial prioritization**.

To provide both practical experience and a clinically relevant learning scenario, this notebook includes two complementary workflows:

* **Workflow A – Standard Annotation Pipeline:** Annotate and prioritize the filtered VCF generated in Notebook 1.
* **Workflow B – Simulated Clinical Cases:** Apply the same workflow to a simulated patient containing a known pathogenic variant, preparing the data for clinical interpretation in the next notebook.

The outputs of both workflows are prioritized annotated VCF files containing candidate variants for downstream phenotype-driven analysis and clinical interpretation.

---

# Learning Objectives

By the end of this notebook, you will be able to:

* Explain the purpose of variant annotation in a germline WES workflow.
* Annotate variants using the **Ensembl Variant Effect Predictor (VEP)**.
* Interpret the biological information added during the annotation process.
* Apply a simple filtering strategy to prioritize candidate variants.
* Inspect and evaluate the resulting candidate variant list.
* Prepare annotated variants for phenotype-driven prioritization and ACMG/AMP-based interpretation in the next notebook.

---

# Overview

Starting from a filtered germline VCF file, this notebook demonstrates how to annotate variants with gene, transcript, protein consequence, population frequency, and other functional annotations using **Ensembl VEP**. The annotated variants are then filtered to retain a manageable set of candidate variants for downstream investigation.

The notebook is designed for **educational purposes** and emphasizes reproducibility, clarity, and understanding of each analysis step. While the workflow follows widely accepted bioinformatics best practices, it is intended as a learning resource and <span style= "color: #ff0000"> **should not be used for clinical diagnosis or medical decision-making**.</span>

By completing both workflows, you will generate prioritized annotated VCF files that are ready for phenotype-driven analysis and clinical interpretation in the subsequent notebooks of this series.


---

**Table of contents**<a id='toc0_'></a>    


- [Introduction](#toc1_)
- [Requirements and Environment Setup](#toc2_)

- [Input Datasets](#toc3_)
  - [Dataset A: Filtered VCF from Notebook 1](#toc3_1_)
  - [Dataset B: Simulated Clinical Case VCF](#toc3_2_)

- [Workflow A: Annotating a Filtered WES VCF](#toc4_)
  - [Load the Input VCF](#toc4_1_)
  - [Variant Annotation with VEP](#toc4_2_)
  - [Filter the Annotated Variants](#toc4_3_)
  - [Inspect the Candidate Variants](#toc4_4_)

- [Workflow B: Simulated Clinical Case Analysis](#toc5_)
  - [Clinical Case 01](#toc5_1_)
    - [Load the Input VCF](#toc5_1_1_)
    - [Annotate the Clinical Case Variants](#toc5_1_2_)
  - [Clinical Case 02 - Comming Soon ...](#toc5_2_)
    
  
- [Summary](#toc6_)
- [References](#toc7_)


<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

---

## <a id='toc1_'></a>[Introduction](#toc0_)

After variant calling and quality filtering, a VCF file contains thousands of genomic variants represented only by their chromosomal positions and reference/alternate alleles. While this information identifies where sequence differences occur, it provides little insight into their potential biological or clinical significance.

Variant annotation is the process of enriching each variant with additional information, such as the affected gene, transcript, predicted functional consequence, protein change, population allele frequency, and known clinical evidence. This additional context enables researchers to move beyond genomic coordinates and begin evaluating which variants may be relevant to a phenotype or disease.

In this notebook, we use the **Ensembl Variant Effect Predictor (VEP)**, one of the most widely used annotation tools in human genomics. VEP compares variants against Ensembl databases and multiple external resources to predict their functional effects and add biologically meaningful annotations to the VCF file.

Following annotation, we apply a straightforward filtering strategy to reduce the number of candidate variants based on commonly used criteria, such as functional consequence, population frequency, and existing clinical annotations. These filtering steps are intended to prioritize variants for further investigation—they do **not** establish clinical pathogenicity.

The final output of this notebook is an annotated and filtered VCF file that serves as the starting point for variant prioritization and ACMG/AMP-based interpretation in the next notebooks of this series.


---

## <a id='toc2_'></a>[Requirements and Environment Setup](#toc0_)

- **Installing required tools**

In [ ]:
!apt-get update
!apt-get install -y cpanminus libssl-dev zlib1g-dev libncurses5-dev libbz2-dev liblzma-dev libperl-dev curl unzip


In [ ]:
!cpanm Module::Build
!cpanm Bio::Perl
!cpanm Set::IntervalTree
!cpanm Scalar::Util::Numeric
!cpanm JSON
!cpanm PerlIO::gzip
!cpanm DBI DBD::mysql
!cpanm Archive::Zip

In [ ]:
!git clone https://github.com/Ensembl/ensembl-vep.git

In [ ]:
cd /content/ensembl-vep

In [ ]:
# Using this verision 
!git checkout release/114

In [ ]:
!perl INSTALL.pl

- **Check the VEP tool**

In [ ]:
!perl /content/ensembl-vep/vep --help

- **download the local annotation database for the GRCh38 reference genome**

It contains information such as:

Gene models
Transcripts
Protein sequences
Variant consequence definitions
Regulatory annotations
Other Ensembl annotation data

* **hg38 reference genome databases**

In [ ]:
# list and get the available reference fasta files for homo sapiens hg38 version
!perl /content/ensembl-vep/INSTALL.pl -a f -s homo_sapiens -g list

In [ ]:
# 24.9 Gb file volume ~20 minutes runtime
!curl -O ftp://ftp.ensembl.org/pub/release-114/variation/indexed_vep_cache/homo_sapiens_vep_114_GRCh38.tar.gz

In [ ]:
# Extract the fata reference .gz file
!gunzip /root/.vep/homo_sapiens/114_GRCh38/Homo_sapiens.GRCh38.dna.toplevel.fa.gz

In [ ]:
# Extract the cache reference .gz file
!tar xzf /content/ensembl-vep/homo_sapiens_vep_114_GRCh38.tar.gz

In [ ]:
# Check the files in the directoy  
ls /root/.vep/homo_sapiens/114_GRCh38/

In [ ]:
#Remove unnecessary  gz file
!rm /content/ensembl-vep/homo_sapiens_vep_114_GRCh38.tar.gz

* **hg19 reference genome databases** (*OPTIONAL FOR WORKFLOW B*)

The simulated clinical cases included in this series are based on the **GRCh37 (hg19)** human genome assembly. Therefore, a separate GRCh37 reference genome is required for annotating these datasets.

This reference genome is used **only for Workflow B** and is independent of the **GRCh38** reference genome used throughout Workflow A. Keeping the two workflows separate ensures that each dataset is annotated using the correct genome assembly while also demonstrating how different reference builds are encountered in real genomic analyses.

> **Note:** The genome assembly of the reference FASTA, annotation resources, and input VCF must always match to ensure accurate variant annotation.


In [ ]:
!curl -O https://ftp.ensembl.org/pub/release-114/variation/indexed_vep_cache/homo_sapiens_vep_114_GRCh37.tar.gz

In [ ]:
!perl /content/ensembl-vep/INSTALL.pl -a f -s homo_sapiens -y GRCh37

In [ ]:
!tar xzf /content/ensembl-vep/homo_sapiens_vep_114_GRCh37.tar.gz

In [ ]:
!gunzip /root/.vep/homo_sapiens/114_GRCh37/Homo_sapiens.GRCh37.75.dna.primary_assembly.fa.gz

In [ ]:
#Remove gz file
!rm /content/ensembl-vep/homo_sapiens_vep_114_GRCh37.tar.gz

In [ ]:
# Check the files in the directoy
!ls /root/.vep/homo_sapiens/114_GRCh37/

- **Project Directories**

To keep the workflow organized and reproducible, all files generated during the analysis are stored in dedicated project directories.

```text
WES_Analysis/
│
├── VCF/
│   └── Final VCF used for annotation
│
├── Annotation/
│   ├── Annotated VCF files
│   └── Annotation reports
│
└── Prioritization/
    └── Filtered annotated VCF files ready for downstream interpretation
```

### Directory Description

* **VCF/** – Stores VCF files generated after variant calling and hard filtering.

* **Annotation/** – Stores VEP-annotated VCF files containing functional and clinical annotations.

* **Prioritization/** – Stores filtered annotated VCF files containing candidate variants

In [ ]:
!mkdir /content/WES_Analysis
!mkdir /content/WES_Analysis/Annotation /content/WES_Analysis/VCF /content/WES_Analysis/Prioritization

In [ ]:
VCF = '/content/WES_Analysis/VCF'
Prioritization = '/content/WES_Analysis/Prioritization'
Annotation ='/content/WES_Analysis/Annotation'

---

## <a id='toc3_'></a>[Input Datasets](#toc0_)

### <a id='toc3_1_'></a>[Dataset A: Filtered VCF from the Germline WES Pipeline](#toc0_)

**Purpose**

This dataset is the output generated from the previous notebook, [**Germline WES Variant Calling**](https://github.com/namia47/WES_Analysis_Tutorial/notebooks/01_Germline_WES_Variant_Calling.ipynb). It contains high-confidence variants that have already passed quality control and hard filtering.

**Goal**

Use this dataset to continue the complete WES analysis workflow by performing:

- Variant annotation
- Population frequency annotation
- Functional consequence annotation
- Clinical annotation
- Preparation for variant prioritization

This option demonstrates how the notebooks are connected as part of a complete end-to-end WES analysis pipeline.

**Recommended for**

- Users following the notebook series from the beginning.
- Anyone who has already completed the Germline WES Variant Calling notebook.

### <a id='toc3_2_'></a>[Dataset B: Simulated Clinical Case VCF](#toc0_)


Dataset B consists of simulated clinical case VCF files designed for educational purposes. Each dataset represents a realistic patient scenario and contains thousands of genomic variants, similar to those obtained from a clinical Whole Exome Sequencing (WES) experiment.

Unlike Dataset A, which originates from the WES analysis performed in Notebook 1, these simulated datasets allow learners to practice variant annotation and clinical interpretation without requiring access to real patient sequencing data.

In this notebook, each clinical case VCF will be annotated using Ensembl Variant Effect Predictor (VEP) and saved as an annotated VCF. These annotated datasets serve as the starting point for **Notebook 3 – Clinical Variant Interpretation**, where they will be combined with phenotype information and analyzed using phenotype-driven prioritization and selected ACMG/AMP evidence to identify the most likely disease-causing variant.

Additional simulated clinical cases will be introduced throughout this series to provide hands-on experience with different genetic disorders, inheritance patterns, and variant interpretation scenarios.

**Recommended for**

- Beginners who want to focus on annotation.
- Workshop participants.
- Users with limited computational resources.
- Anyone interested in practicing annotation and variant interpretation without running the complete WES pipeline.

---